# **Local sensitivity of interflow/baseflow partitioning to the impedance factor $\Omega$**
## Parametrization: impedance factor, forward-mode AD (`jax.jvp`) w.r.t. $\Omega$

Mirrors the setup of `Constant_imp_omega10.ipynb` exactly (same grid, BCs, spinup), but instead of
loading a cached, hardcoded-$\Omega$ interpolation table, it builds the table live via
`utils.sensitivity.build_soil_type_impedance` and differentiates the whole 60-day solve w.r.t.
$\Omega$ using `jax.jvp` (see `utils/sensitivity.py`).

**Cost note**: forward-mode AD through the full solve costs roughly 2x a normal forward run.
`t_end` below defaults to a short 5-day dry run -- confirm it runs cleanly and the sensitivity
curves look sane before extending to the full 60 days (`60*24*60*60`) used in the paper run.

Author: Anne Hermann   \
anne.hermann@uni-tuebingen.de

In [1]:
import sys
sys.path.append('../')
sys.path.append('../../../')

import jax.numpy as jnp
import matplotlib.pyplot as plt
import pickle as pkl

from utils.geometry import Geom
from utils import plotting_utils
from utils import sensitivity

cm = 1/2.54

In [3]:
omega0 = 10.0  # nominal impedance factor to linearise around (permaFoam_omega10 uses Omega=10)

spinup_folder = "../../../output/Simulations_Paper/ERA5_Sweden/spinup/tl1/"
spinup_filename = "spinup_permaFoam_omega10.pkl"

save_folder = "../../../output/Simulations_Paper/Constant/sensitivity/"
save_file = "omega_sensitivity_permaFoam_omega10.pkl"

In [5]:
dz = 0.025       # grid spacing [m]
dx = 0.025       # grid spacing [m]
ztot = 1         # height of the cross-section [m]
xtot = 10        # length of the domain [m]
slope = -0.1     # slope of the domain [-]

geom = Geom(dx, dz, 0, xtot, -ztot, 0, slope)
geom.create_grid()
geom.set_sparsity_pattern()

===============   Creating grid   =============== 
DONE: created grid with  400 x 40 cells.
===============   Calculating coupled sparsity pattern   =============== 
DONE: calculated coupled sparsity pattern with 316480 nonzero entries.


In [6]:
dt = 60                    # initial time step [s]
t0 = 0
t_end = 5*24*60*60          # end time [s] -- SHORT DRY RUN; use 60*24*60*60 for the full paper-scale run
maxdt = 1.5*60*60           # maximum time step [s]
save_t = 2*60*60            # save time [s]
geom.time_discretization(dt, t0, t_end, maxdt, save_t)

===============   Creating time discretization   =============== 
DONE: created time discretization.


In [7]:
# Sandy soil parameters
soil_type = {1: {}, 2: {}}
soil = jnp.ones((geom.nz, geom.nx))
soil_type[1]["alpha"] = 14.5
soil_type[1]["Ksat"] = 8.25e-5
soil_type[1]["N"] = 2.68
soil_type[1]["Theta_r"] = 0.045
soil_type[1]["Theta_s"] = 0.43

# Loamy soil parameters
indices = jnp.where((geom.Z <= -.25 + slope*geom.X) & (geom.Z >= -.5 + slope*geom.X))
soil = soil.at[indices].set(2)
soil_type[2]["alpha"] = 1.6
soil_type[2]["Ksat"] = 6.94e-7
soil_type[2]["N"] = 1.37
soil_type[2]["Theta_r"] = 0.034
soil_type[2]["Theta_s"] = 0.46

geom.soil = soil

# discretize S -> hc and T, same grids as the production interpolation tables
Se = jnp.geomspace(1e-6, 1-1e-6, int(1e5))
T = jnp.append(jnp.append(jnp.arange(-30, -1, 1), jnp.arange(-1, 0, 1e-3)), jnp.array([0.01, 5]))

Build a concrete (non-traced) table at $\Omega=\Omega_0$ just to set the initial condition and sanity-plot
$\theta_l$/$k_{rel}$, exactly like the production notebook does. `sensitivity.omega_sensitivity` below rebuilds
its own (traced) copy internally -- this one is only used for setup.

In [8]:
soil_type0 = sensitivity.build_soil_type_impedance(Se, T, soil_type, omega0)
geom.set_soilType(soil, soil_type0)

KeyboardInterrupt: 

In [ ]:
t_bc = jnp.array([0, t_end])
q_inf = jnp.array([2e-8, 2e-8])
T_surface = jnp.array([-2, -2])
geom.set_surface_bc(t_bc, q_inf, T_surface)

In [ ]:
with open(spinup_folder + spinup_filename, 'rb') as pickle_file:
    data = pkl.load(pickle_file)
    theta_l_total = data['theta_l_total']
    theta_i_total = data['theta_i_total']
    T_total = data['T_total']
    h_total = data['h_total']
    h_c_total = data['h_c_total']

Theta_l = theta_l_total[-1]
Theta_i = theta_i_total[-1]
h_c = h_c_total[-1]
htot = h_total[-1]
T_init = T_total[-1]
geom.set_initial_conditions(htot, h_c, Theta_l, Theta_i, T_init)

theta_l = jnp.where(geom.soil == 1,
                     geom.soil_type[1]["theta_l_interp"](h_c.flatten(), T_init.flatten()).reshape(h_c.shape),
                     geom.soil_type[2]["theta_l_interp"](h_c.flatten(), T_init.flatten()).reshape(h_c.shape))
plotting_utils.plot_cellValues(geom.X, geom.Z, theta_l, colormap="Blues", title=r"Initial $\theta_l$", colorbar_label=r'$\theta_l$ [-]')

krel = jnp.where(geom.soil == 1,
                  geom.soil_type[1]["k_rel_interp"](h_c.flatten(), T_init.flatten()).reshape(h_c.shape),
                  geom.soil_type[2]["k_rel_interp"](h_c.flatten(), T_init.flatten()).reshape(h_c.shape))
plotting_utils.plot_cellValues(geom.X, geom.Z, krel, colormap="Reds", title=r"Initial $k_{rel}$", colorbar_label=r'$k_{rel}$ [-]')

geom.set_initial_conditions(htot, h_c, theta_l, 0*theta_l, T_init)

In [ ]:
rho_l = 1000; rho_i = 917; rho_s = 2650; rho_a = 1200
c_s = 800; c_l = 4180; c_i = 2100; c_a = 1000
H_f = 3.34e5
geom.set_thermal_params(rho_l, rho_i, rho_s, rho_a, c_l, c_i, c_s, c_a, H_f)

restol_rich = 1e-4
restol_T = 1e-4
maxiter = 20
miniter = 5
geom.set_newton_params(restol_rich, restol_T, maxiter, miniter)

## Run: forward-mode sensitivity w.r.t. $\Omega$

One `jax.jvp` call runs the full coupled solve once and returns, in the same pass, the exit-flux
time series *and* their derivatives w.r.t. $\Omega$ -- no finite-difference re-runs needed.

In [ ]:
time_array, (q_top, q_bottom, q_surf), (d_qtop, d_qbottom, d_qsurf) = sensitivity.omega_sensitivity(
    geom, Se, T, soil_type, omega0
)
time_days = time_array / 24 / 60 / 60

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(30*cm, 9*cm))
for ax, q, label in zip(axes, [q_top, q_bottom, q_surf], ["Interflow ($q_{top}$)", "Baseflow ($q_{bottom}$)", "Surface runoff ($q_{surface}$)"]):
    ax.plot(time_days, q, linewidth=2, color=plotting_utils.persian_red)
    ax.set_title(label)
    ax.set_xlabel("time [d]")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].set_ylabel(r"$q$ [m$^2$/s]")
fig.suptitle(r"Flux trajectories at $\Omega_0=$" + f"{omega0}")
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(30*cm, 9*cm))
for ax, dq, label in zip(axes, [d_qtop, d_qbottom, d_qsurf], ["Interflow ($q_{top}$)", "Baseflow ($q_{bottom}$)", "Surface runoff ($q_{surface}$)"]):
    ax.plot(time_days, dq, linewidth=2, color=plotting_utils.carolina_blue)
    ax.axhline(0, color="grey", linestyle="--", linewidth=1)
    ax.set_title(label)
    ax.set_xlabel("time [d]")7	'
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].set_ylabel(r"$d q / d\Omega$")
fig.suptitle(r"Local sensitivity to $\Omega$ at $\Omega_0=$" + f"{omega0}")
fig.tight_layout()
fig.savefig(save_folder + "omega_sensitivity_permaFoam_omega10.png", dpi=300)

In [ ]:
db = {
    "omega0": omega0,
    "time_array": time_array,
    "q_top": q_top, "q_bottom": q_bottom, "q_surf": q_surf,
    "d_qtop_domega": d_qtop, "d_qbottom_domega": d_qbottom, "d_qsurf_domega": d_qsurf,
}
with open(save_folder + save_file, 'wb') as pickle_file:
    pkl.dump(db, pickle_file)